# 03 — Evaluation & Model Comparison

This notebook evaluates trained models on the NuminaMath-CoT test set and
provides a head-to-head comparison between the SFT and RL (GRPO) models.

**Metrics:** Exact-match accuracy on `\boxed{}` answers with symbolic equality fallback.

## 1. Setup

In [ ]:
# Clone the repo (replace with your GitHub URL)
!git clone https://github.com/YOUR_USERNAME/math-rl-tuning.git
%cd math-rl-tuning

# Install the package
!pip install -e ".[viz]" --quiet
!pip install bitsandbytes latex2sympy2 --quiet

In [ ]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, mount_google_drive

cfg = load_config()
setup_hf_token()
mount_google_drive()

## 2. Configure Model Paths

In [ ]:
# Point these to your saved adapters (local or Google Drive)
SFT_ADAPTER_PATH = cfg.paths.sft_output_dir
RL_ADAPTER_PATH = cfg.paths.grpo_output_dir

# Or use Drive paths:
# SFT_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"
# RL_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/grpo"

NUM_SAMPLES = 50  # Number of test examples to evaluate

## 3. Load Test Data

In [ ]:
from math_rl_tuning.data import prepare_test_data

test_ds = prepare_test_data(cfg)
print(f"Test examples: {len(test_ds)}")
print(f"Sample problem: {test_ds[0]['problem'][:200]}...")

## 4. Evaluate a Single Model

Use this to evaluate just one model (SFT or RL).

In [ ]:
from math_rl_tuning.evaluation import evaluate_adapter

sft_df, sft_acc = evaluate_adapter(
    adapter_path=SFT_ADAPTER_PATH,
    test_dataset=test_ds,
    cfg=cfg,
    num_samples=NUM_SAMPLES,
    model_name="SFT",
)

print(f"\nSFT Accuracy: {sft_acc:.2%}")
sft_df.head(10)

## 5. Head-to-Head Comparison (SFT vs RL)

In [ ]:
from math_rl_tuning.evaluation import compare_models, save_results

results = compare_models(
    sft_adapter_path=SFT_ADAPTER_PATH,
    rl_adapter_path=RL_ADAPTER_PATH,
    test_dataset=test_ds,
    cfg=cfg,
    num_samples=NUM_SAMPLES,
)

# Save results to disk
save_results(results, cfg.paths.eval_output_dir)

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Bar chart: accuracy comparison
fig, ax = plt.subplots(figsize=(6, 4))
models = ["SFT", "RL (GRPO)"]
accuracies = [results["sft_accuracy"], results["rl_accuracy"]]
colors = ["#4C72B0", "#DD8452"]

bars = ax.bar(models, accuracies, color=colors, width=0.5)
ax.set_ylabel("Accuracy")
ax.set_title("SFT vs RL (GRPO) — Math Accuracy")
ax.set_ylim(0, 1.0)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{acc:.1%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()

In [ ]:
# Show examples where RL improved over SFT
sft_df = results["sft_results"]
rl_df = results["rl_results"]

if sft_df is not None and rl_df is not None:
    comparison = sft_df[["problem_snippet", "ground_truth"]].copy()
    comparison["SFT_answer"] = sft_df["predicted"]
    comparison["SFT_correct"] = sft_df["is_correct"]
    comparison["RL_answer"] = rl_df["predicted"]
    comparison["RL_correct"] = rl_df["is_correct"]

    improved = comparison[(~comparison["SFT_correct"]) & comparison["RL_correct"]]
    regressed = comparison[comparison["SFT_correct"] & (~comparison["RL_correct"])]

    print(f"Cases where RL FIXED SFT errors: {len(improved)}")
    print(f"Cases where RL REGRESSED:        {len(regressed)}")
    print()

    if not improved.empty:
        print("=== RL Improvements ===")
        display(improved)

## 7. Cleanup

In [ ]:
from math_rl_tuning.utils import clean_memory
clean_memory()